In [1]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send, interrupt, Command
from typing import TypedDict
import subprocess, textwrap
from openai import OpenAI
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator, base64
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

llm = init_chat_model("openai:gpt-4o-mini")

class State(TypedDict):
    video_file: str
    audio_file: str
    transcription: str
    summaries: Annotated[list[str], operator.add]
    thumbnail_prompts: Annotated[list[str], operator.add]
    thumbnail_sketches: Annotated[list[str], operator.add]
    final_summary: str
    user_feedback: str
    chosen_prompt: str

In [2]:
def extract_audio(state: State):
    # ffmpeg
    output_file = state["video_file"].replace("mp4", "mp3")
    command = [
        "ffmpeg",
        "-i",
        state["video_file"], 
        "-filter:a",
        "atempo=2.0",
        "-y",
        output_file
    ]
    subprocess.run(command)
    return {
        "audio_file": output_file,
    }

def transcribe_audio(state: State):
    # use audio file
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=audio_file,
            language="ko",
            prompt="창의성, 사고력, AI" # hint for Model
        )
        return {
            "transcription": transcription
        }

def dispatch_summarizers(state: State):
    transcription = state["transcription"]
    chunks = []
    for idx, chunk in enumerate(textwrap.wrap(transcription, 200)):
        chunks.append({
            "id": idx+1,
            "chunk": chunk,
        })
    return [Send("summarize_chunk", chunk) for chunk in chunks]

def summarize_chunk(chunk):
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Please summarize the following text with Only Korean.

        Text: {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return {
        "summaries": [summary]
    }

def mega_summary(state: State):
    all_summaries = "\n".join(state["summaries"])
    prompt = f"""
    You are given multiple summaries of different chunks from a video transcription.
    Please create a comprehensive final summary that combines all the key points.
    Individual summaries:
    {all_summaries}
    """

    response = llm.invoke(prompt)

    return {
        "final_summary": response.content,
    }

def dispatch_artists(state: State):
    return [
        Send("generate_thumbnails", {"id": i, "summary": state["final_summary"]})
        for i in [1, 2, 3]
    ]

def generate_thumbnails(args):
    id = args["id"]
    summary = args["summary"]

    prompt = f"""
    Based on this video summary, create a detailed visual prompt for a YouTube thumbnail.

    Create a detailed prompt for generating a thumbnail image that would attract viewers. Include:
        - Main visual elements
        - Color scheme
        - Text overlay suggestions
        - Overall composition
    
    Summary: {summary}
    """
    response = llm.invoke(prompt)
    thumbnail_prompt = response.content

    client = OpenAI()
    result = client.images.generate(
        model="gpt-image-1",
        prompt=thumbnail_prompt,
        quality="low",
        moderation="low",
        size="auto"
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)
    filename = f"thumbnail_{id}.jpg"

    with open(filename, "wb") as file:
        file.write(image_bytes)

    return {
        "thumbnail_prompts": [thumbnail_prompt],
        "thumbnail_sketches": [filename]
    }

def human_feedback(state: State):
    answer = interrupt({
        "chosen_thumbnail": "Which thumbnail do yo like the most?",
        "feedback": "Provide any feedback or changes you'd like for the final thumbnail."
    })
    user_feedback = answer["user_feedback"]
    chosen_prompt = answer["chosen_prompt"]

    return {
        "user_feedback": user_feedback,
        "chosen_prompt": state["thumbnail_prompts"][chosen_prompt-1]
    }

def generate_hd_thumbnail(state: State):
    chosen_prompt = state["chosen_prompt"]
    user_feedback = state["user_feedback"]

    prompt = f"""
    You are a professional YouTube thumbnail designer. Take this original thumbnail prompt and create an enhanced version that incorporates the user's specific feedback.

    ORIGINAL PROMPT:
    {chosen_prompt}

    USER FEEDBACK TO INCORPORATE:
    {user_feedback}

    Create an enhanced prompt that:
        1. Maintains the core concept from the original prompt
        2. Specifically addresses and implements the user's feedback requests
        3. Adds professional YouTube thumbnail specifications:
            - High contrast and bold visual elements
            - Clear focal points that draw the eye
            - Professional lighting and composition
            - Optimal text placement and readability with generous padding from edges
            - Colors that pop and grab attention
            - Elements that work well at small thumbnail sizes
            - IMPORTANT: Always ensure adequate white space/padding between any text and the image borders
    """

    response = llm.invoke(prompt)
    final_thumbnail_prompt = response.content

    client = OpenAI()
    result = client.images.generate(
        model="gpt-image-1",
        prompt=final_thumbnail_prompt,
        quality="high",
        moderation="low",
        size="auto"
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)
    filename = f"thumbnail_final.jpg"

    with open(filename, "wb") as file:
        file.write(image_bytes)

In [3]:
graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)
graph_builder.add_node("mega_summary", mega_summary)
graph_builder.add_node("generate_thumbnails", generate_thumbnails)
graph_builder.add_node("human_feedback", human_feedback)
graph_builder.add_node("generate_hd_thumbnail", generate_hd_thumbnail)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges("transcribe_audio", dispatch_summarizers , ["summarize_chunk"])
graph_builder.add_edge("summarize_chunk", "mega_summary")
graph_builder.add_conditional_edges("mega_summary", dispatch_artists, ["generate_thumbnails"])
graph_builder.add_edge("generate_thumbnails", "human_feedback")
graph_builder.add_edge("human_feedback", "generate_hd_thumbnail")
graph_builder.add_edge("generate_hd_thumbnail", END)

graph = graph_builder.compile(checkpointer=memory)

In [4]:
config = {
    "configurable": {
        "thread_id": "1",
    },
}

In [5]:
graph.invoke(
    { "video_file": "asset/one_minute_video.mp4" }, 
    config=config,
)

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

{'video_file': 'asset/one_minute_video.mp4',
 'audio_file': 'asset/one_minute_video.mp3',
 'transcription': '딱 1분만 집중해서 들어봐 요즘 청소년들의 AI 꽝손이 심각하다는 말 들어봤지? 아니 AI 쓰고 인생 좀 편하게 살 수 있는 거 아닌가 했는데 부분이 이게 내 발달에 부정적인 영향을 미치는 과도적인 이유가 있더라고 미국의 MIT 미디어랩에서 사람들을 모아다가 SA를 작성하는 시험을 했거든 일단 팀을 세 개로 나눴는데 한 팀은 무조건 머리만 굴려서 SA를 쓰게 했고 다른 한 팀은 공부 같은 검색엔진 나머지 한 팀은 그냥 우리의 예상처럼 채 GPT가 된 LRM을 쓰게 했지 그리고 SA를 작성하는 동안만 우리가 어떻게 돌아가는지 확인하기 위해 뇌파를 확인했어 그래서 우리 머리로만 승부한 뒤 뇌의 연결성이 가장 유미했던 거야 아주 강력하고 분산된 네트워크로 온 뇌를 잘 굴린 거지 그리고 검색엔진을 쓴 그룹은 중간 정도 LRM을 쓴 그룹은 뇌의 연결성이 가장 약했어 우리를 제일 안 썼다는 거야 당연히 SA에서 자신의 주장 또한 제일 부족했고 시간이 조금만 지나도 본인이 대체 붙었는지 제대로 기억해내지 못했지 그리고 이렇게 LRM을 사용한 그룹은 점차 언어 행동 수준을 창의성에 대해서 계속적으로 저조한 성과를 나타냈어 언론은 직접 사고하는 노력을 기울이잖아 AI가 생각에 유존한다면 우리의 사고력은 점차 태화한다는 거야 안 가서는 대대로 된 사고가 안 된다는 건데 형 이제 학교 공부는 버거워지고 시험 문제도 못 그리겠지 차라리 스스로 생각하고 이후에 AI의 도움을 받는 게 훨씬 좋아 이 편집은 뭐 옛날 연애하는 방법 AI랑 상관없어\n',
 'summaries': ['[Chunk 1] 요즘 청소년들이 AI 사용에 의존하면서 부정적인 영향을 받고 있다는 연구 결과가 있다. MIT 미디어랩에서 실시한 시험에서는 팀을 나누어 AI 없이 SA를 작성하는 방식과 AI를 사용하는 방식을 비교

In [6]:
# snapshot = graph.get_state(config)

# snapshot.next

response = {
    "user_feedback": "Please refine the figure drawing to make it look a bit more natural.",
    "chosen_prompt": 3,
}

graph.invoke(
    Command(resume=response),
    config=config,
)

{'video_file': 'asset/one_minute_video.mp4',
 'audio_file': 'asset/one_minute_video.mp3',
 'transcription': '딱 1분만 집중해서 들어봐 요즘 청소년들의 AI 꽝손이 심각하다는 말 들어봤지? 아니 AI 쓰고 인생 좀 편하게 살 수 있는 거 아닌가 했는데 부분이 이게 내 발달에 부정적인 영향을 미치는 과도적인 이유가 있더라고 미국의 MIT 미디어랩에서 사람들을 모아다가 SA를 작성하는 시험을 했거든 일단 팀을 세 개로 나눴는데 한 팀은 무조건 머리만 굴려서 SA를 쓰게 했고 다른 한 팀은 공부 같은 검색엔진 나머지 한 팀은 그냥 우리의 예상처럼 채 GPT가 된 LRM을 쓰게 했지 그리고 SA를 작성하는 동안만 우리가 어떻게 돌아가는지 확인하기 위해 뇌파를 확인했어 그래서 우리 머리로만 승부한 뒤 뇌의 연결성이 가장 유미했던 거야 아주 강력하고 분산된 네트워크로 온 뇌를 잘 굴린 거지 그리고 검색엔진을 쓴 그룹은 중간 정도 LRM을 쓴 그룹은 뇌의 연결성이 가장 약했어 우리를 제일 안 썼다는 거야 당연히 SA에서 자신의 주장 또한 제일 부족했고 시간이 조금만 지나도 본인이 대체 붙었는지 제대로 기억해내지 못했지 그리고 이렇게 LRM을 사용한 그룹은 점차 언어 행동 수준을 창의성에 대해서 계속적으로 저조한 성과를 나타냈어 언론은 직접 사고하는 노력을 기울이잖아 AI가 생각에 유존한다면 우리의 사고력은 점차 태화한다는 거야 안 가서는 대대로 된 사고가 안 된다는 건데 형 이제 학교 공부는 버거워지고 시험 문제도 못 그리겠지 차라리 스스로 생각하고 이후에 AI의 도움을 받는 게 훨씬 좋아 이 편집은 뭐 옛날 연애하는 방법 AI랑 상관없어\n',
 'summaries': ['[Chunk 1] 요즘 청소년들이 AI 사용에 의존하면서 부정적인 영향을 받고 있다는 연구 결과가 있다. MIT 미디어랩에서 실시한 시험에서는 팀을 나누어 AI 없이 SA를 작성하는 방식과 AI를 사용하는 방식을 비교